In [1]:
import re
import torch
import torch.nn as nn
import torch.optim as optim

from collections import Counter
from torch.utils.data import Dataset, DataLoader

In [2]:
sentences = [
    "i love machine learning",
    "i love deep learning",
    "machine learning is interesting",
    "deep learning is powerful",
    "i enjoy learning new things",
    "machine learning is useful",
    "i love artificial intelligence",
    "artificial intelligence is interesting",
    "deep learning uses neural networks",
    "neural networks learn patterns",
    "machine learning uses data",
    "i enjoy machine learning",
    "learning new things is useful",
    "neural networks are powerful",
    "i love neural networks",
    "artificial intelligence uses data"
]

print("Number of sentences:", len(sentences))

Number of sentences: 16


## Tokenization

Tokenization converts text into smaller units called tokens.

For this project, each token will be a word.

Example:

Sentence: `i love machine learning`

Tokens:

`["i", "love", "machine", "learning"]`

The model cannot directly process ordinary words as text. We will map each word to a numerical ID.


In [3]:
def tokenize(sentence):
    return re.findall(r"\b\w+\b", sentence.lower())


tokenized_sentences = [
    tokenize(sentence)
    for sentence in sentences
]

for sentence in tokenized_sentences[:5]:
    print(sentence)

['i', 'love', 'machine', 'learning']
['i', 'love', 'deep', 'learning']
['machine', 'learning', 'is', 'interesting']
['deep', 'learning', 'is', 'powerful']
['i', 'enjoy', 'learning', 'new', 'things']


In [4]:
tokenized_sentences

[['i', 'love', 'machine', 'learning'],
 ['i', 'love', 'deep', 'learning'],
 ['machine', 'learning', 'is', 'interesting'],
 ['deep', 'learning', 'is', 'powerful'],
 ['i', 'enjoy', 'learning', 'new', 'things'],
 ['machine', 'learning', 'is', 'useful'],
 ['i', 'love', 'artificial', 'intelligence'],
 ['artificial', 'intelligence', 'is', 'interesting'],
 ['deep', 'learning', 'uses', 'neural', 'networks'],
 ['neural', 'networks', 'learn', 'patterns'],
 ['machine', 'learning', 'uses', 'data'],
 ['i', 'enjoy', 'machine', 'learning'],
 ['learning', 'new', 'things', 'is', 'useful'],
 ['neural', 'networks', 'are', 'powerful'],
 ['i', 'love', 'neural', 'networks'],
 ['artificial', 'intelligence', 'uses', 'data']]

## Vocabulary

A vocabulary is the collection of distinct tokens the model knows.

For example:

`["i", "love", "machine", "learning"]`

We assign each unique word an integer ID.

Example:

| Word     | ID |
| -------- | -: |
| i        |  0 |
| love     |  1 |
| machine  |  2 |
| learning |  3 |

The actual vocabulary in our project will contain all unique words from the dataset.

We will also reserve an ID for unknown words so that prediction inputs containing unfamiliar words can be handled safely.


In [5]:
word_counts = Counter(
    word                                    # 1. What you want to collect (the individual word)
    for sentence in tokenized_sentences     # 2. The outer loop (pulls each sentence)
    for word in sentence                    # 3. The inner loop (pulls each word from that sentence)
)

vocab = ["<UNK>"] + sorted(word_counts.keys())

word_to_id = {
    word: index
    for index, word in enumerate(vocab)
}

id_to_word = {
    index: word
    for word, index in word_to_id.items()
}

vocab_size = len(vocab)

print("Vocabulary size:", vocab_size)
print("\nWord to ID mapping:")

for word, index in word_to_id.items():
    print(word, ":", index)

Vocabulary size: 22

Word to ID mapping:
<UNK> : 0
are : 1
artificial : 2
data : 3
deep : 4
enjoy : 5
i : 6
intelligence : 7
interesting : 8
is : 9
learn : 10
learning : 11
love : 12
machine : 13
networks : 14
neural : 15
new : 16
patterns : 17
powerful : 18
things : 19
useful : 20
uses : 21


In [6]:
word_counts

Counter({'learning': 10,
         'i': 6,
         'machine': 5,
         'is': 5,
         'love': 4,
         'neural': 4,
         'networks': 4,
         'deep': 3,
         'artificial': 3,
         'intelligence': 3,
         'uses': 3,
         'interesting': 2,
         'powerful': 2,
         'enjoy': 2,
         'new': 2,
         'things': 2,
         'useful': 2,
         'data': 2,
         'learn': 1,
         'patterns': 1,
         'are': 1})

In [7]:
vocab

['<UNK>',
 'are',
 'artificial',
 'data',
 'deep',
 'enjoy',
 'i',
 'intelligence',
 'interesting',
 'is',
 'learn',
 'learning',
 'love',
 'machine',
 'networks',
 'neural',
 'new',
 'patterns',
 'powerful',
 'things',
 'useful',
 'uses']

In [8]:
word_to_id

{'<UNK>': 0,
 'are': 1,
 'artificial': 2,
 'data': 3,
 'deep': 4,
 'enjoy': 5,
 'i': 6,
 'intelligence': 7,
 'interesting': 8,
 'is': 9,
 'learn': 10,
 'learning': 11,
 'love': 12,
 'machine': 13,
 'networks': 14,
 'neural': 15,
 'new': 16,
 'patterns': 17,
 'powerful': 18,
 'things': 19,
 'useful': 20,
 'uses': 21}

## Creating Training Examples

The RNN will learn to predict the next word from the preceding words.

Suppose the sentence is:

`i love machine learning`

We can create these training examples:

| Input            | Target     |
| ---------------- | ---------- |
| `i`              | `love`     |
| `i love`         | `machine`  |
| `i love machine` | `learning` |

The input contains the preceding words, and the target is the next word.

We convert each word into its vocabulary ID.

For example, if:

`i = 5`, `love = 12`, `machine = 18`, `learning = 15`

Then the examples become:

Input: `[5]`, target: `12`

Input: `[5, 12]`, target: `18`

Input: `[5, 12, 18]`, target: `15`

These numerical values are illustrative; the actual IDs come from the vocabulary built in the code.

Our dataset class will return a variable-length input sequence and one target ID for each example.


In [11]:
tokenized_sentences

[['i', 'love', 'machine', 'learning'],
 ['i', 'love', 'deep', 'learning'],
 ['machine', 'learning', 'is', 'interesting'],
 ['deep', 'learning', 'is', 'powerful'],
 ['i', 'enjoy', 'learning', 'new', 'things'],
 ['machine', 'learning', 'is', 'useful'],
 ['i', 'love', 'artificial', 'intelligence'],
 ['artificial', 'intelligence', 'is', 'interesting'],
 ['deep', 'learning', 'uses', 'neural', 'networks'],
 ['neural', 'networks', 'learn', 'patterns'],
 ['machine', 'learning', 'uses', 'data'],
 ['i', 'enjoy', 'machine', 'learning'],
 ['learning', 'new', 'things', 'is', 'useful'],
 ['neural', 'networks', 'are', 'powerful'],
 ['i', 'love', 'neural', 'networks'],
 ['artificial', 'intelligence', 'uses', 'data']]

In [14]:
training_examples = []

for tokens in tokenized_sentences:
    token_ids = [word_to_id[word]
        for word in tokens]
    for i in range(1, len(token_ids)):
        input_sequence = token_ids[:i]
        target_word = token_ids[i]

        training_examples.append(
            (input_sequence, target_word)
        )

print("Total training examples:", len(training_examples))

for input_sequence, target_word in training_examples[:10]:
    input_words = [
        id_to_word[index]
        for index in input_sequence
    ]

    print(
        "Input:", input_words,
        "-> Target:", id_to_word[target_word]
    )

Total training examples: 51
Input: ['i'] -> Target: love
Input: ['i', 'love'] -> Target: machine
Input: ['i', 'love', 'machine'] -> Target: learning
Input: ['i'] -> Target: love
Input: ['i', 'love'] -> Target: deep
Input: ['i', 'love', 'deep'] -> Target: learning
Input: ['machine'] -> Target: learning
Input: ['machine', 'learning'] -> Target: is
Input: ['machine', 'learning', 'is'] -> Target: interesting
Input: ['deep'] -> Target: learning


## Dataset and DataLoader

A PyTorch Dataset defines how individual training examples are retrieved.

A DataLoader groups examples into batches.

For example, a batch size of 4 means the model processes four training examples at a time.

Because our input sequences have different lengths, we pad shorter sequences with a special padding ID.

Padding lets us create rectangular tensors suitable for batching.

We will use a separate padding ID so the model can distinguish padding from real words.


In [15]:
PAD_ID = vocab_size

class NextWordDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def collate_batch(batch):
    sequences, targets = zip(*batch)

    max_length = max(
        len(sequence)
        for sequence in sequences
    )

    padded_sequences = []

    for sequence in sequences:
        padding_length = max_length - len(sequence)

        padded_sequence = (
            [PAD_ID] * padding_length
            + sequence
        )

        padded_sequences.append(padded_sequence)

    inputs = torch.tensor(
        padded_sequences,
        dtype=torch.long
    )

    targets = torch.tensor(
        targets,
        dtype=torch.long
    )

    return inputs, targets


dataset = NextWordDataset(training_examples)

dataloader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_batch
)

for inputs, targets in dataloader:
    print("Input shape:", inputs.shape)
    print("Target shape:", targets.shape)
    print("Inputs:\n", inputs)
    print("Targets:\n", targets)
    break

Input shape: torch.Size([8, 4])
Target shape: torch.Size([8])
Inputs:
 tensor([[22, 22,  6, 12],
        [22, 22, 22,  6],
        [22, 22, 22, 15],
        [22, 22,  6,  5],
        [22, 22,  2,  7],
        [11, 16, 19,  9],
        [22,  6, 12, 15],
        [22, 22,  2,  7]])
Targets:
 tensor([ 4,  5, 14, 13,  9, 20, 14, 21])


In [18]:
class NextWordRNN(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=32,
        hidden_size=64
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size + 1,
            embedding_dim=embedding_dim,
            padding_idx=PAD_ID
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            vocab_size
        )

    def forward(self, x):
        lengths = (x != PAD_ID).sum(dim=1)

        embedded = self.embedding(x)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, hidden = self.rnn(packed)

        final_hidden = hidden[-1]

        logits = self.fc(final_hidden)

        return logits

In [19]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = NextWordRNN(
    vocab_size=vocab_size,
    embedding_dim=32,
    hidden_size=64
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Device:", device)
print(model)

Device: cpu
NextWordRNN(
  (embedding): Embedding(23, 32, padding_idx=22)
  (rnn): RNN(32, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=22, bias=True)
)


In [20]:
epochs = 20
for epoch in range(epochs):
    model.train()

    total_loss = 0.0

    for inputs, targets in dataloader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        logits = model(inputs)

        loss = criterion(
            logits,
            targets
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(dataloader)

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch + 1}/{epochs}], "
            f"Loss: {average_loss:.4f}"
        )

Epoch [10/20], Loss: 2.4349
Epoch [20/20], Loss: 1.9126


## Prediction

To predict the next word:

1. Convert the input text into tokens.
2. Convert the tokens into vocabulary IDs.
3. Handle unknown words.
4. Convert the sequence into a tensor.
5. Pass it to the trained model.
6. Select the vocabulary word with the highest output logit.


In [21]:
def predict_next_word(text):
    model.eval()

    tokens = tokenize(text)

    if not tokens:
        raise ValueError("Please provide a non-empty input.")

    token_ids = [
        word_to_id.get(word, word_to_id["<UNK>"])
        for word in tokens
    ]

    inputs = torch.tensor(
        [token_ids],
        dtype=torch.long,
        device=device
    )

    with torch.inference_mode():
        logits = model(inputs)

        predicted_id = torch.argmax(
            logits,
            dim=1
        ).item()

    return id_to_word[predicted_id]


print("Input: i love")
print("Predicted next word:", predict_next_word("i love"))

print("\nInput: machine learning")
print("Predicted next word:", predict_next_word("machine learning"))

print("\nInput: neural networks")
print("Predicted next word:", predict_next_word("neural networks"))

Input: i love
Predicted next word: learning

Input: machine learning
Predicted next word: is

Input: neural networks
Predicted next word: patterns


In [27]:
def generate_text(start_text, num_words=5):
    model.eval()

    generated_words = tokenize(start_text)

    if not generated_words:
        raise ValueError("Please provide a non-empty starting text.")

    for _ in range(num_words):
        input_ids = [
            word_to_id.get(word, word_to_id["<UNK>"])
            for word in generated_words
        ]

        inputs = torch.tensor(
            [input_ids],
            dtype=torch.long,
            device=device
        )

        with torch.inference_mode():
            logits = model(inputs)

            next_id = torch.argmax(
                logits,
                dim=1
            ).item()

        next_word = id_to_word[next_id]

        if next_word == "<UNK>":
            break

        generated_words.append(next_word)

    return " ".join(generated_words)


print(generate_text("i love", num_words=3))

i love learning learning things


In [28]:
import torch
import torch.nn as nn

embedding = nn.Embedding(100, 32)

rnn = nn.RNN(
    input_size=32,
    hidden_size=16,
    batch_first=True
)

inputs = torch.tensor([[1, 5, 8, 12, 20, 25]])

embedded = embedding(inputs)

output, hidden = rnn(embedded)

print("Input shape:", inputs.shape)
print("Embedding shape:", embedded.shape)
print("RNN output shape:", output.shape)
print("Hidden state shape:", hidden.shape)

Input shape: torch.Size([1, 6])
Embedding shape: torch.Size([1, 6, 32])
RNN output shape: torch.Size([1, 6, 16])
Hidden state shape: torch.Size([1, 1, 16])
